## Traditional Attempt at classification (Random Forest & XGBoost)

This is a first attempt at classification of the combined dataset using Random Forest.

Reading in the data

In [13]:
#Bad things happen when you uncomment these lines
# Beware
#df = pd.read_parquet('../data/MERGED_LCS.parquet')
#df

Reducing the precision of the timestamps and flux values so my kernel stops crashing

In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
import numpy as np
import pandas as pd

infile = "../data/MERGED_LCS.parquet"
outfile = "../data/MERGED_LCS_reduced.parquet"

def round_array(x, ndigits=5):
    if x is None:
        return None
    arr = np.asarray(x, dtype=np.float32)
    return np.round(arr, ndigits).tolist()

pf = pq.ParquetFile(infile)
writer = None

for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    # Convert directly from Arrow to a normal Python dict
    data = batch.to_pydict()

    # Build the dataframe explicitly, including KIC
    df = pd.DataFrame({
        "time": data["time"],
        "flux": data["flux"],
        "Class": data["Class"],
        "KIC": data["KIC"],
    })

    df["time"] = df["time"].apply(round_array)
    df["flux"] = df["flux"].apply(round_array)

    # Save this batch
    table = pa.Table.from_pandas(df, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(outfile, table.schema, compression="zstd")

    writer.write_table(table)

if writer is not None:
    writer.close()

In [5]:
batch = next(pf.iter_batches(batch_size=1, columns=["time", "flux", "Class", "KIC"]))
df = batch.to_pandas(ignore_metadata=True)
print(df.columns)
print(df.head())

Index(['time', 'flux', 'Class', 'KIC'], dtype='object')
                                                time  \
0  [131.51271468758932, 131.5331494016791, 131.55...   

                                                flux      Class     KIC  
0  [1.0392150925008297, 1.0383358659303283, 1.038...  CONFIRMED  757450  


In [2]:
df = pd.read_parquet("../data/MERGED_LCS_reduced.parquet")

In [3]:
df

,time,flux,Class,KIC
0,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0392199754714966, 1.0383399724960327, 1.038...",CONFIRMED,757450
1,"[352.39654541015625, 352.43743896484375, 352.4...","[1.019569993019104, 1.018839955329895, 1.01899...",FALSE POSITIVE,892772
2,"[120.539306640625, 120.55975341796875, 120.580...","[1.0488500595092773, 1.0523099899291992, 1.051...",CANDIDATE,1025986
3,"[131.51271057128906, 131.53314208984375, 131.5...","[1.0509400367736816, 1.0503300428390503, 1.050...",FALSE POSITIVE,1026032
4,"[120.53929901123047, 120.55973815917969, 120.5...","[1.059939980506897, 1.0591700077056885, 1.0589...",CONFIRMED,1026957
...,...,...,...,...
18877,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9961699843406677, 0.9965299963951111, 0.996...",VARIABLE STAR,202140012
18878,"[1940.0084228515625, 1940.02880859375, 1940.04...","[0.9897400140762329, 0.9907699823379517, 0.991...",VARIABLE STAR,202140013
18879,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0229099988937378, 1.014359951019287, 1.0042...",FALSE POSITIVE,202140059
18880,"[1940.0089111328125, 1940.029296875, 1940.0498...","[1.0080900192260742, 1.0085899829864502, 1.009...",FALSE POSITIVE,202140094


## Preprocessing

In [8]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from scipy.ndimage import median_filter

def sigma_clip_mad(t, f, sigma=5.0):
    """
    Robust outlier removal using MAD.
    Returns filtered time and flux arrays.
    """
    if len(f) == 0:
        return t, f

    med = np.median(f)
    mad = np.median(np.abs(f - med))

    if mad == 0:
        return t, f

    robust_sigma = 1.4826 * mad
    keep = np.abs(f - med) <= sigma * robust_sigma
    return t[keep], f[keep]

def normalize_by_median(f):
    """
    Normalize flux by its median.
    """
    med = np.median(f)
    if med == 0:
        return f
    return f / med

def detrend_with_median_filter(t, f, window_frac=0.1):
    """
    Detrend by interpolating onto a uniform grid and subtracting a median-filter trend.
    Works best after normalization.
    """
    n = len(f)
    if n < 10:
        return f

    # Interpolate onto a uniform grid
    grid = np.linspace(t.min(), t.max(), n, dtype=np.float32)
    f_interp = np.interp(grid, t, f).astype(np.float32)

    # Median filter window size
    win = max(5, int(n * window_frac))
    if win >= n:
        win = n - 1
    if win % 2 == 0:
        win -= 1
    if win < 5:
        return f

    trend = median_filter(f_interp, size=win, mode="nearest")
    trend_at_t = np.interp(t, grid, trend).astype(np.float32)

    # Avoid divide-by-zero
    trend_at_t = np.where(np.abs(trend_at_t) < 1e-8, 1.0, trend_at_t)

    # Since normalized flux is usually around 1, division is okay here
    return f / trend_at_t


def preprocess_lightcurve(time, flux, round_digits=5, sigma=5.0, detrend=True,n_points = 200):
    """
    Clean one light curve:
    - convert to float32
    - drop NaNs/infs
    - sort by time
    - round values
    - normalize by median
    - sigma-clip outliers
    - optionally detrend
    Returns cleaned time and flux arrays.
    """
    t = np.asarray(time, dtype=np.float32)
    f = np.asarray(flux, dtype=np.float32)

    # Drop NaN/inf
    mask = np.isfinite(t) & np.isfinite(f)
    t = t[mask]
    f = f[mask]

    if len(t) == 0:
        return t, f

    # Sort by time
    order = np.argsort(t)
    t = t[order]
    f = f[order]

    # Reduce precision
    t = np.round(t, round_digits)
    f = np.round(f, round_digits)

    # Normalize
    f = normalize_by_median(f)

    # Remove outliers
    t, f = sigma_clip_mad(t, f, sigma=sigma)

    # Detrend
    if detrend and len(t) >= 10:
        f = detrend_with_median_filter(t, f)
    
    # normalize time → [0,1]
    t = (t - t.min()) / (t.max() - t.min() + 1e-8)

    # fixed grid
    t_new = np.linspace(0, 1, n_points)

    # interpolate
    f_new = np.interp(t_new, t, f)


    return t.astype(np.float32), f.astype(np.float32)

In [9]:
infile = "../data/MERGED_LCS.parquet"
pf = pq.ParquetFile(infile)

cleaned_rows = []

for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    df = batch.to_pandas(ignore_metadata=True)

    for t, f, label, kic in zip(df["time"], df["flux"], df["Class"], df["KIC"]):
        t_clean, f_clean = preprocess_lightcurve(t, f)

        cleaned_rows.append({
            "time": t_clean,
            "flux": f_clean,
            "Class": label,
            "KIC": kic
        })

cleaned_df = pd.DataFrame(cleaned_rows)

: 

In [7]:
cleaned_df

,time,flux,Class,KIC
0,"[131.51271, 131.53314, 131.55359, 131.57402, 1...","[1.0, 0.9991532, 0.999567, 0.99896085, 0.99857...",CONFIRMED,757450
1,"[352.39655, 352.43744, 352.4579, 352.47833, 35...","[1.0, 0.99928397, 0.9994312, 0.999235, 1.00026...",FALSE POSITIVE,892772
2,"[120.53931, 120.55975, 120.58018, 120.60062, 1...","[1.0, 1.0032988, 1.0029174, 1.0024979, 1.00206...",CANDIDATE,1025986
3,"[131.51271, 131.53314, 131.55359, 131.57402, 1...","[1.0, 0.99941957, 0.9991436, 0.99845845, 0.998...",FALSE POSITIVE,1026032
4,"[120.5393, 120.55974, 120.58017, 120.6006, 120...","[1.0, 0.9992736, 0.99903774, 0.99936795, 0.999...",CONFIRMED,1026957
...,...,...,...,...
18877,"[1940.0084, 1940.0288, 1940.0491, 1940.0697, 1...","[1.0, 1.0000043, 1.0001451, 1.0000622, 0.99980...",VARIABLE STAR,202140012
18878,"[1940.0084, 1940.0288, 1940.0493, 1940.0697, 1...","[1.0, 1.0010407, 1.001657, 1.0019196, 1.001879...",VARIABLE STAR,202140013
18879,"[1940.0089, 1940.0293, 1940.0498, 1940.0702, 1...","[1.0, 0.99164146, 0.98174816, 0.97545236, 0.97...",FALSE POSITIVE,202140059
18880,"[1940.0089, 1940.0293, 1940.0498, 1940.0702, 1...","[1.0, 1.0004959, 1.0018847, 0.9955659, 0.99950...",FALSE POSITIVE,202140094


In [1]:
import pyarrow.parquet as pq
pf = pq.ParquetFile("../data/MERGED_LCS.parquet")
print(pf.metadata.num_rows)
print(pf.metadata.num_row_groups)

18882
1


In [4]:
for batch in pf.iter_batches(batch_size=16, columns=["time", "flux", "Class", "KIC"]):
    df = batch.to_pandas(ignore_metadata=True)
    print("batch loaded:", df.shape)
    break

batch loaded: (16, 4)
